# Tugas Praktikum 3: Classification dengan Ensemble Method & Class Imbalance
Zico Diego Rio Ramadhonny (5054251023)


## Persiapan Program


In [1]:
import os
import kagglehub
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict

from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN
from imblearn.under_sampling import RandomUnderSampler, NearMiss, TomekLinks

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score


## Persiapan Dataset


In [2]:
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print("Path to dataset files:", path)

all_files = [os.path.join(path, item) for item in os.listdir(path)]
csv_path = None
for item in all_files:
    filename = os.path.basename(item)
    if os.path.isfile(item) and os.path.splitext(filename)[1] == ".csv":
        csv_path = item
        print(f"\nFound the CSV file at {item}.")
        break
    
if csv_path is None:
    raise FileNotFoundError("The CSV path couldn't be found.")

df = pd.read_csv(csv_path)
print("CSV loaded successfully.")


Path to dataset files: /home/zico/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3
Found the CSV file at /home/zico/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3/creditcard.csv.
CSV loaded successfully.


## EDA Awal


### Visualisasi Informasi Dataset


In [3]:
print("Statistik deskriptif:")
display(df.describe())
display(df.info())
print("Shape:", df.shape)

print("\n\nHead:")
display(df.head(10))

print("\n\nHistogram")
display(df.hist(figsize=(30, 16), bins=10))


Statistik deskriptif:


<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-null  

None


Shape: (284807, 31)
Head:


Histogram


array([[<Axes: title={'center': 'Time'}>, <Axes: title={'center': 'V1'}>,
        <Axes: title={'center': 'V2'}>, <Axes: title={'center': 'V3'}>,
        <Axes: title={'center': 'V4'}>, <Axes: title={'center': 'V5'}>],
       [<Axes: title={'center': 'V6'}>, <Axes: title={'center': 'V7'}>,
        <Axes: title={'center': 'V8'}>, <Axes: title={'center': 'V9'}>,
        <Axes: title={'center': 'V10'}>, <Axes: title={'center': 'V11'}>],
       [<Axes: title={'center': 'V12'}>, <Axes: title={'center': 'V13'}>,
        <Axes: title={'center': 'V14'}>, <Axes: title={'center': 'V15'}>,
        <Axes: title={'center': 'V16'}>, <Axes: title={'center': 'V17'}>],
       [<Axes: title={'center': 'V18'}>, <Axes: title={'center': 'V19'}>,
        <Axes: title={'center': 'V20'}>, <Axes: title={'center': 'V21'}>,
        <Axes: title={'center': 'V22'}>, <Axes: title={'center': 'V23'}>],
       [<Axes: title={'center': 'V24'}>, <Axes: title={'center': 'V25'}>,
        <Axes: title={'center': 'V26'}>, <

## Visualizasi Matrix Korelasi


In [4]:
corr_data = df.corr()
corr_data_rounded = corr_data.round(2)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(20, 16))
ax = sns.heatmap(corr_data_rounded, cmap="Blues", annot=True)
plt.show()


## Pencetakan Informasi Mengenai Korelasi Fitur terhadap Target


In [5]:
sort_class_corr = corr_data["Class"].sort_values(key=lambda x: abs(x), ascending=False).drop(index=["Class"])
print("Atribut dengan korelasi tertinggi terhadap target:")
for feat, corr_val in sort_class_corr.items():
    print(f"- {feat}:", corr_val)


Atribut dengan korelasi tertinggi terhadap target:
- V17: -0.32648106724371595
- V14: -0.3025436958044028
- V12: -0.26059292487722485
- V10: -0.21688294364103206
- V16: -0.19653894030401736
- V3: -0.19296082706741602
- V7: -0.18725659151430013
- V11: 0.15487564474394433
- V4: 0.13344748623900718
- V18: -0.11148525388904133
- V1: -0.10134729859508294
- V9: -0.0977326860740807
- V5: -0.09497429899144802
- V2: 0.0912886503446179
- V6: -0.04364316069996494
- V21: 0.04041338061057565
- V19: 0.034783013036515056
- V20: 0.020090324196974638
- V8: 0.019875123914794384
- V27: 0.01757972818951348
- Time: -0.012322570929245588
- V28: 0.00953604091623609
- V24: -0.007220906715952582
- Amount: 0.005631753006768552
- V13: -0.004569778799461309
- V26: 0.0044553975012833254
- V15: -0.004223402267856675
- V25: 0.0033077055972996613
- V23: -0.0026851557402507824
- V22: 0.0008053175052983079


# Praproses Data


## Visualisasi Distribusi Kelas


In [6]:
percentage_count = df["Class"].value_counts(normalize=True)
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(20, 8))
print(percentage_count)

# 1
df["Class"].hist(ax=ax[0])
ax[0].set_title("Distribusi Jumlah Label")
ax[0].set_xlabel("Class")
ax[0].set_ylabel("Jumlah")

ax[1].barh(percentage_count.values, percentage_count.keys(), 0.1)
ax[1].set_title("Distribusi Persentase Label")
ax[1].set_xlabel("Persentase (%)")
ax[1].set_ylabel("Class")

for i, count in enumerate([df["Class"].value_counts(), percentage_count]):
    # plt.text(x, y, s,)
    for x, y in count.items():
        y_coor = y
        if i == 1:
            y_coor = 1 - y_coor
            x = abs(x - 1)
        
        ax[i].text(x, y_coor, str(y) if i == 0 else f"{y:.3f}", ha="left")

plt.show()


Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64


## Pembagian Data ke Train dan Test


In [7]:
RANDOM_STATE = 42
scaler = StandardScaler()

X = df.drop(columns=["Class"])
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RANDOM_STATE, stratify=y)

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## Inisialisasi Metode Sampling untuk Class Imbalance


In [8]:
sampling_dict = {
    "oversampling": {
        "SMOTE": SMOTE(random_state=RANDOM_STATE),
        "RandomOverSampler": RandomOverSampler(random_state=RANDOM_STATE),
        "ADASYN": ADASYN(random_state=RANDOM_STATE)
    },
    "undersampling": {
        "RandomUnderSampler": RandomUnderSampler(random_state=RANDOM_STATE),
        "NearMiss": NearMiss(),
        "TomekLinks": TomekLinks()
    }
}

transformed_train_dict = {
    type_sampling: {
        method: method_object.fit_resample(X_train, y_train) for method, method_object in sampling_type_dict.items()
    } for type_sampling, sampling_type_dict in sampling_dict.items()
}


## Visualisasi Distribusi Kelas Setelah Penanganan Class Imbalance


In [9]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(16, 9), layout="constrained")
all_transformed_train = transformed_train_dict["oversampling"] | transformed_train_dict["undersampling"]
ax = ax.flatten()

for i, (method, (_, transformed_y)) in enumerate(all_transformed_train.items()):
    random_state_print = "" if method in ["NearMiss", "TomekLinks"] else f" (random_state={RANDOM_STATE})"
    
    ax[i].set_title(f"Distribusi Label Setelah {method}{random_state_print}")
    unique_count = np.unique(transformed_y, return_counts=True)
    ax[i].bar(unique_count[0], unique_count[1], 0.1)
    ax[i].set_xlabel("Class")
    ax[i].set_ylabel("Jumlah")
    ax[i].set_xticks(unique_count[0])
    
    for cls_id, count in zip(unique_count[0], unique_count[1]):
        ax[i].text(cls_id, count, str(count), ha="center", fontsize=8)
    
plt.show()


## Membangun Model Klasifikasi


In [10]:
def run_model_dict_dfs(model_dict: dict, fit_data: dict) -> None:
    if not model_dict:
        return
    
    for key, item in model_dict.items():
        if isinstance(item, dict):
            run_model_dict_dfs(item, fit_data[key])
            continue
        
        if key not in fit_data:
            X_train_curr, y_train_curr = X_train, y_train
            item.fit(X_train, y_train)
        else:
            X_train_curr, y_train_curr = fit_data[key]
            item.fit(X_train_curr, y_train_curr)
            
        print(f"Fitted {key} model ({item} with {len(X_train_curr)} data).")

class_weight_by_percentage = 1 - df["Class"].value_counts(normalize=True)
class_weight_dict = {cls_id: w_i for cls_id, w_i in class_weight_by_percentage.items()}

baseline = RandomForestClassifier(random_state=RANDOM_STATE)
cost_sensitive = RandomForestClassifier(class_weight=class_weight_dict, random_state=RANDOM_STATE)

sampling_model = {
    type_sampling: {
        method: RandomForestClassifier(random_state=RANDOM_STATE) for method in sampling_type_dict.keys()
    } for type_sampling, sampling_type_dict in sampling_dict.items()
}

model_collection_dict = {
    "Baseline": baseline,
    "Cost-sensitive/Weighted": cost_sensitive
} | sampling_model

run_model_dict_dfs(model_collection_dict, transformed_train_dict)

# Thirdteen minutes of execution time


Fitted Baseline model (RandomForestClassifier(random_state=42) with 213605 data).
Fitted Cost-sensitive/Weighted model (RandomForestClassifier(class_weight={0: 0.0017274856306200848,
                                     1: 0.9982725143693799},
                       random_state=42) with 213605 data).
Fitted SMOTE model (RandomForestClassifier(random_state=42) with 426472 data).
Fitted RandomOverSampler model (RandomForestClassifier(random_state=42) with 426472 data).
Fitted ADASYN model (RandomForestClassifier(random_state=42) with 426461 data).
Fitted RandomUnderSampler model (RandomForestClassifier(random_state=42) with 738 data).
Fitted NearMiss model (RandomForestClassifier(random_state=42) with 738 data).
Fitted TomekLinks model (RandomForestClassifier(random_state=42) with 213587 data).


# Evaluasi Model


In [21]:
def run_test_dfs(model_dict: dict, X_val: pd.DataFrame, y_val: pd.Series) -> dict:
    if not model_dict:
        return
    
    result_dict = dict()
    for key, item in model_dict.items():
        if isinstance(item, dict):
            result_dict[key] = run_test_dfs(item, X_val, y_val)
            continue
    
        metric_score_dict = {}
        prediction = item.predict(X_val)
        probability = item.predict_proba(X_val)[:, 1]
        
        metric_score_dict["classification_report"] = classification_report(y_val, prediction, output_dict=True)
        metric_score_dict["roc_auc_score"] = roc_auc_score(y_val, probability)
        metric_score_dict["accuracy_score"] = accuracy_score(y_val, prediction)
        metric_score_dict["confusion_matrix"] = confusion_matrix(y_val, prediction)
        result_dict[key] = metric_score_dict
    
        print(f"Evaluated {key} model ({item}).")
        
    return result_dict
        
eval_res = run_test_dfs(model_collection_dict, X_test, y_test)
eval_res_all_methods = {
    key: item for key, item in eval_res.items() if key in ["Baseline", "Cost-sensitive/Weighted"]
} | eval_res["oversampling"] | eval_res["undersampling"]

eval_cleaned_dict = defaultdict(list)
for key, item in eval_res_all_methods.items():
    macro_avg_data = eval_res_all_methods[key]["classification_report"]["macro avg"]
    roc_auc_res = eval_res_all_methods[key]["roc_auc_score"]
    acc_score = eval_res_all_methods[key]["accuracy_score"]
    
    eval_cleaned_dict["Metode"].append(key)
    for metric, metric_score in macro_avg_data.items():
        if metric == "support":
            continue
        
        eval_cleaned_dict[metric.capitalize()].append(metric_score)
        
    eval_cleaned_dict["AUC-ROC"].append(roc_auc_res)
    eval_cleaned_dict["Accuracy"].append(acc_score)    
    
display(pd.DataFrame(eval_cleaned_dict))
print("Akurasi saja tidak cukup untuk menilai performa sebuah model karena sangat didominasi oleh class dominan")
print("yang lebih banyak dilihat oleh model dan berpotensi menimbulkan bias. Metrik lain, seperti precision, re-")
print("call, F1-score, AUC-ROC, dan lain sebagainya, dapat menangani hal tersebut dengan melihat distribusi hasil")
print("prediksi berdasarkan tiap class maupun hubungan antara true positive dan false positive.")

fig, ax = plt.subplots(nrows=3, ncols=3, figsize=(16, 12), layout="constrained")
ax = ax.flatten()
for i, (key, item) in enumerate(eval_res_all_methods.items()):
    sns.heatmap(item["confusion_matrix"], ax=ax[i], cmap="Blues", annot=True, fmt="d")
    ax[i].set_title(f"Confusion Matrix Hasil {key}")
ax[-1].axis("off")
plt.show()


Evaluated Baseline model (RandomForestClassifier(random_state=42)).
Evaluated Cost-sensitive/Weighted model (RandomForestClassifier(class_weight={0: 0.0017274856306200848,
                                     1: 0.9982725143693799},
                       random_state=42)).
Evaluated SMOTE model (RandomForestClassifier(random_state=42)).
Evaluated RandomOverSampler model (RandomForestClassifier(random_state=42)).
Evaluated ADASYN model (RandomForestClassifier(random_state=42)).
Evaluated RandomUnderSampler model (RandomForestClassifier(random_state=42)).
Evaluated NearMiss model (RandomForestClassifier(random_state=42)).
Evaluated TomekLinks model (RandomForestClassifier(random_state=42)).


Akurasi saja tidak cukup untuk menilai performa sebuah model karena sangat didominasi oleh class dominan
yang lebih banyak dilihat oleh model dan berpotensi menimbulkan bias. Metrik lain, seperti precision, re-
call, F1-score, AUC-ROC, dan lain sebagainya, dapat menangani hal tersebut dengan melihat distribusi hasil
prediksi berdasarkan tiap class maupun hubungan antara true positive dan false positive.
